# 04 — Pediatric HIV drug resistance in the dolutegravir era

## An MCH-centered Stanford HIVDB case study

This notebook was developed to extend the **MCH Office Hours** presentation **“HIV Drug Resistance in MCH Populations”** by **Spencer Lloyd, MPH, MD; Laura Broyles, MD; and Jason Bacha, MD** (CDC/GHC/DGHT/HCTB, August 19, 2026).

The presentation highlights a difficult MCH problem: children living with HIV can accumulate resistance across several generations of antiretroviral therapy, while the populations sent for resistance testing are often highly selected for virologic non-suppression, treatment experience, or other risk factors. Stanford HIVDB provides a useful interpretation layer for understanding how individual resistance mutations and mutation combinations may affect specific antiretroviral drugs.

> **Educational scope:** All child profiles below are fictional. This notebook is not patient-care guidance and does not replace national treatment guidelines, expert consultation, or a complete clinical history.

### Learning goals
By the end of this notebook, you should be able to:
- preserve the correct denominator when reading pediatric DTG-resistance estimates;
- connect pediatric regimen history with likely resistance “footprints”;
- recognize MCH-relevant NRTI and INSTI mutation patterns;
- understand why Stanford HIVDB uses drug-specific scores and combination rules rather than a binary mutation flag; and
- distinguish what a genotype can tell you from the adherence, maternal, social, and programmatic information it cannot tell you.


## 1. Start with the MCH signal — and preserve the denominator

Lloyd, Broyles, and Bacha summarize pediatric DTG-resistance results from several studies and preliminary CADRE analyses. Importantly, these estimates come from **selected pediatric populations**, generally children with virologic non-suppression or failure and sufficient specimens for genotyping. They are **not** prevalence estimates among all children receiving ART.

The presentation's preliminary CADRE summary includes the illustrative country-level values below. Inclusion criteria and study designs differ, so these values should not be pooled or interpreted as directly comparable national prevalence estimates.

**Source:** Lloyd S, Broyles L, Bacha J. *HIV Drug Resistance in MCH Populations*. MCH Office Hours. CDC/GHC/DGHT/HCTB; August 19, 2026.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

cadre = pd.DataFrame([
    {"country": "Ethiopia", "dtg_resistance_pct": 6.3, "years": "2021–2022"},
    {"country": "Malawi", "dtg_resistance_pct": 16.0, "years": "2022–2023"},
    {"country": "South Sudan", "dtg_resistance_pct": 5.3, "years": "2022–2023"},
    {"country": "Tajikistan", "dtg_resistance_pct": 6.7, "years": "2023"},
    {"country": "Tanzania", "dtg_resistance_pct": 8.3, "years": "2023"},
    {"country": "Uganda — Round 1", "dtg_resistance_pct": 4.1, "years": "2021–2022"},
    {"country": "Uganda — Round 2", "dtg_resistance_pct": 8.4, "years": "2023–2024"},
])
cadre

In [ ]:
plot_df = cadre.sort_values("dtg_resistance_pct")
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(plot_df["country"], plot_df["dtg_resistance_pct"])
ax.set_xlabel("DTG resistance in the selected pediatric analysis population (%)")
ax.set_title("Presentation-derived pediatric DTG-resistance estimates")
for i, v in enumerate(plot_df["dtg_resistance_pct"]):
    ax.text(v + 0.25, i, f"{v:g}%", va="center")
plt.tight_layout()
plt.show()

### Denominator thought experiment

Suppose a program has 1,000 children receiving ART. If 10% meet a virologic non-suppression criterion and 8% of that selected subgroup have DTG resistance, then **8% among the VNS subgroup is not 8% of all children on ART**. The values below are hypothetical and illustrate only the denominator issue.


In [ ]:
children_on_art = 1000
vns_fraction = 0.10
resistance_among_vns = 0.08

children_with_vns = children_on_art * vns_fraction
children_with_dtg_resistance = children_with_vns * resistance_among_vns
population_fraction = children_with_dtg_resistance / children_on_art

pd.DataFrame({
    "Quantity": [
        "Children on ART",
        "Children meeting VNS criterion",
        "Children with DTG resistance in VNS subgroup",
        "Implied fraction of all children on ART",
    ],
    "Value": [
        children_on_art,
        children_with_vns,
        children_with_dtg_resistance,
        f"{population_fraction:.1%}",
    ],
})

In this hypothetical example, **8% among children with VNS corresponds to 0.8% of all children on ART**. The actual relationship depends on viral-load coverage, selection into testing, amplification success, adherence patterns, and other program factors. This denominator discipline is essential when translating resistance studies into MCH program decisions.


## 2. Pediatric regimen history creates an HIVDR “memory”

The MCH presentation emphasizes that many children are heavily treatment-experienced and may have been exposed to NNRTIs, multiple NRTI backbones, protease inhibitors, raltegravir, and later dolutegravir. A genotype collected today can therefore reflect **years of prior selection pressure** rather than only the current regimen.

The synthetic profiles below are teaching examples designed around mutation families highlighted in the presentation and Stanford HIVDB resistance notes.


In [ ]:
profiles = pd.DataFrame([
    {
        "profile": "A — VNS without listed major DRM",
        "history": "Persistent viremia; no teaching mutations listed below",
        "RT_mutations": [],
        "IN_mutations": [],
    },
    {
        "profile": "B — ABC/3TC footprint",
        "history": "Historical ABC/3TC exposure",
        "RT_mutations": ["L74V", "M184V"],
        "IN_mutations": [],
    },
    {
        "profile": "C — NRTI + DTG-era pathway",
        "history": "Long treatment history; NRTI DRMs plus an INSTI DRM",
        "RT_mutations": ["K65R", "M184V"],
        "IN_mutations": ["R263K"],
    },
    {
        "profile": "D — G118R pathway",
        "history": "NRTI resistance plus a DTG-associated INSTI pattern",
        "RT_mutations": ["M184V"],
        "IN_mutations": ["G118R", "E138K"],
    },
    {
        "profile": "E — Q148 pathway",
        "history": "Major Q148 pathway with an accessory mutation",
        "RT_mutations": ["M184V"],
        "IN_mutations": ["Q148H", "G140S"],
    },
])
profiles

## 3. Mutation families that resonate with the MCH presentation

The presentation specifically highlights relationships between **NRTI mutations (M184V and K65R)** and **INSTI mutations (R263K and G118R)**, and its virologic exploration slide lists additional pediatric mutations such as Q148H/K/R and N155H.

Stanford HIVDB adds mechanistic detail:
- **M184V/I** strongly reduces 3TC/FTC susceptibility and also reduces viral replication fitness.
- **K65R** can reduce susceptibility to tenofovir, abacavir, and 3TC/FTC.
- **L74V/I** is strongly associated with ABC exposure and often occurs with M184V.
- **R263K** is a major nonpolymorphic INSTI DRM selected during DTG exposure and alone produces modest reductions in DTG/BIC/CAB susceptibility.
- **G118R** is another major DTG-associated INSTI pathway.
- **Q148H/K/R** and **N155H** are major INSTI pathways whose effects can be amplified by accompanying accessory mutations such as G140S or E138K.

**Stanford sources:** HIVDB NRTI and INSTI resistance notes and HIVDB release notes.


In [ ]:
mutation_reference = pd.DataFrame([
    {"mutation": "M184V", "gene": "RT", "class": "NRTI DRM", "teaching_signal": "3TC/FTC resistance; NRTI-backbone history"},
    {"mutation": "K65R", "gene": "RT", "class": "NRTI DRM", "teaching_signal": "TDF/ABC/3TC-FTC effects"},
    {"mutation": "L74V", "gene": "RT", "class": "NRTI DRM", "teaching_signal": "ABC-associated pathway"},
    {"mutation": "R263K", "gene": "IN", "class": "Major INSTI DRM", "teaching_signal": "DTG-era pathway"},
    {"mutation": "G118R", "gene": "IN", "class": "Major INSTI DRM", "teaching_signal": "DTG-era pathway"},
    {"mutation": "Q148H", "gene": "IN", "class": "Major INSTI DRM", "teaching_signal": "Q148 pathway"},
    {"mutation": "N155H", "gene": "IN", "class": "Major INSTI DRM", "teaching_signal": "N155 pathway"},
    {"mutation": "E138K", "gene": "IN", "class": "Accessory INSTI DRM", "teaching_signal": "Can amplify major pathways"},
    {"mutation": "G140S", "gene": "IN", "class": "Accessory INSTI DRM", "teaching_signal": "Common Q148 companion"},
])
mutation_reference

## 4. Visualize mutation combinations, not only “resistant: yes/no”

For MCH program science, the pattern often matters more than a single flag. Which children show only historical NRTI signals? Which show a major INSTI DRM? Which show both a compromised NRTI backbone and an INSTI pathway?


In [ ]:
all_mutations = mutation_reference["mutation"].tolist()
rows = []
for _, row in profiles.iterrows():
    present = set(row["RT_mutations"] + row["IN_mutations"])
    item = {"profile": row["profile"]}
    for mutation in all_mutations:
        item[mutation] = int(mutation in present)
    rows.append(item)

matrix = pd.DataFrame(rows).set_index("profile")
fig, ax = plt.subplots(figsize=(10, 4.8))
ax.imshow(matrix.values, aspect="auto", vmin=0, vmax=1)
ax.set_xticks(range(len(matrix.columns)), matrix.columns, rotation=45, ha="right")
ax.set_yticks(range(len(matrix.index)), matrix.index)
ax.set_title("Synthetic pediatric HIVDR mutation patterns")
for i in range(matrix.shape[0]):
    for j in range(matrix.shape[1]):
        if matrix.iloc[i, j]:
            ax.text(j, i, "●", ha="center", va="center", fontsize=14)
plt.tight_layout()
plt.show()

## 5. Why Stanford HIVDB is more informative than a binary mutation flag

HIVDB assigns **drug-specific mutation penalty scores**, applies combination rules, and maps the resulting total to five interpretation levels: susceptible, potential low-level, low-level, intermediate, and high-level resistance.

A useful MCH teaching example is **L74V + M184V**, because Lloyd, Broyles, and Bacha explicitly raise NRTI-backbone history and ABC recycling as pediatric analytical questions. Stanford documents an abacavir example in which L74V contributes 30 points, M184V contributes 15 points, and the L74V+M184V combination adds another 15 points. The total is 60, which maps to high-level resistance in the HIVDB algorithm.

This is an **algorithm demonstration**, not a recommendation about whether ABC should be continued, recycled, or replaced in a particular child.


In [ ]:
def hivdb_level(score):
    if score < 10:
        return "Susceptible"
    if score < 15:
        return "Potential low-level resistance"
    if score < 30:
        return "Low-level resistance"
    if score < 60:
        return "Intermediate resistance"
    return "High-level resistance"

abc_example = pd.DataFrame([
    {"component": "L74V", "ABC_penalty": 30},
    {"component": "M184V", "ABC_penalty": 15},
    {"component": "L74V + M184V combination rule", "ABC_penalty": 15},
])
score = abc_example["ABC_penalty"].sum()
abc_example, score, hivdb_level(score)

## 6. A copy-ready Stanford HIVDB mutation-list exercise

Take synthetic Profile C: **K65R + M184V + R263K**. A learner can open Stanford's **HIVDB by mutations** tool and enter these mutations to inspect the current drug-specific interpretation.

1. Open https://hivdb.stanford.edu/hivdb/by-mutations/
2. Enter the reverse-transcriptase mutations **K65R M184V** and integrase mutation **R263K** in the appropriate fields.
3. Compare the drug-specific interpretation across NRTIs and INSTIs.
4. Read the mutation comments and notice that the interpretation is not equivalent to “three mutations = three resistant drugs.”
5. Repeat with **M184V + G118R + E138K** or **M184V + Q148H + G140S** and compare how a major INSTI pathway plus an accessory mutation changes the pattern.

Because HIVDB rules are updated over time, the live Stanford output should be treated as the current interpretation source rather than hard-coding every drug score into this notebook.


## 7. What the genotype does *not* explain

The MCH presentation is especially strong on this point. Lloyd, Broyles, and Bacha identify clinical correlates and programmatic factors that a sequence alone cannot resolve: prior virologic failure, duration on ART, adherence challenges, caregiver changes, psychosocial stressors, higher viral load at switch, limited access to viral-load testing, and ART stockouts. They also emphasize maternal ART history and pregnancy/breastfeeding context when thinking about infant resistance.

A genotype can help answer **what viral variants are present** and how they may affect drug susceptibility. It cannot, by itself, tell you **why** those variants emerged, whether drug exposure was adequate, whether a dose was appropriate for weight, whether medicines were available, or whether a caregiver could consistently administer them.


In [ ]:
layers = pd.DataFrame([
    {"layer": "Genotype / HIVDB", "questions": "Which DRMs are present? What drug-specific resistance does HIVDB infer?"},
    {"layer": "Treatment history", "questions": "Which regimens, failures, interruptions, and prior exposures occurred?"},
    {"layer": "Child / caregiver", "questions": "Palatability, school, disclosure, caregiver changes, mental health, adherence?"},
    {"layer": "Maternal / perinatal", "questions": "Maternal ART, breastfeeding exposure, prophylaxis, timing of infection?"},
    {"layer": "Program / system", "questions": "VL access, turnaround time, stockouts, dosing tools, follow-up continuity?"},
])
layers

## 8. Three analytical questions for MCH presenters

The presentation closes with pediatric-specific questions that are well suited to a future Stanford-linked analytic workflow:

**A. Do children with NRTI resistance have different INSTI mutation patterns?**
Stratify genotypes by M184V/I, K65R, TAMs, and ABC-associated mutations; compare R263K, G118R, Q148, and N155 pathways.

**B. Does regimen history explain mutation clusters better than age alone?**
Link genotype interpretation to prior NNRTI, PI, raltegravir, and DTG exposure, plus duration on ART and history of failure.

**C. What is different about infants and children with possible perinatal or breastfeeding drug exposure?**
Separate pretreatment resistance from acquired resistance and incorporate maternal ART/prophylaxis history whenever available.

These questions illustrate where HIVDB interpretation can become one component of a richer MCH analytic dataset.


## 9. Take-home message

For an MCH audience, the most compelling use of Stanford HIVDB is not simply to label a genotype “resistant.” It is to connect **a child's treatment journey** with **specific mutation pathways**, **drug-specific interpretation**, and **the denominator and selection process that produced the tested population**.

A useful mental model is:

**viral load signal → selected child for genotyping → mutation pattern → HIVDB drug-specific interpretation → treatment/maternal/adherence/program context → MCH question**

That final context is where the presentation by Lloyd, Broyles, and Bacha adds the most value: pediatric HIVDR is simultaneously a virologic problem, a treatment-history problem, a caregiver/adherence problem, and a health-system problem.


## References and source attribution

1. **Lloyd S, Broyles L, Bacha J.** *HIV Drug Resistance in MCH Populations.* MCH Office Hours. CDC/GHC/DGHT/HCTB. August 19, 2026. Presentation supplied with this project.
2. Stanford University HIV Drug Resistance Database (HIVDB). https://hivdb.stanford.edu/
3. Stanford HIVDB Release Notes. https://hivdb.stanford.edu/page/release-notes/
4. Stanford HIVDB NRTI Resistance Notes. https://hivdb.stanford.edu/dr-summary/resistance-notes/NRTI/
5. Stanford HIVDB INSTI Resistance Notes. https://hivdb.stanford.edu/dr-summary/resistance-notes/INSTI/
6. Stanford HIVDB Analysis by Mutations. https://hivdb.stanford.edu/hivdb/by-mutations/

**Provenance note:** Numeric pediatric CADRE values in this notebook are transcribed from the Lloyd/Broyles/Bacha presentation for educational visualization. They are not reclassified as Stanford HIVDB surveillance estimates. Stanford HIVDB is used here for resistance-interpretation concepts and mutation-level teaching.
